# Векторный поиск вакансий

Общий site config связывает TEI/E5-base и выбранную физическую коллекцию Qdrant.
Notebook показывает чистую подготовку запроса, документ для модели, последовательный
аудит tokenizer, явный vector/native body, поиск и CSV batch.

Откройте notebook в kernel существующего Python 3.12+ окружения проекта.
Jupyter предоставляется окружением пользователя. Подготовьте site config по README;
TEI и Qdrant должны быть доступны. Notebook не запускает Docker.
Ключи берутся из `TEI_API_KEY` и `QDRANT_API_KEY` окружения kernel. Не вставляйте их
в ячейки/overrides и не выводите resolved config. Outputs могут содержать документы
разрешённого исследовательского корпуса; поставляемый notebook сохранён без них.


In [ ]:
import json
import sys
from pathlib import Path
from uuid import uuid4

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs/vector.yaml").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate

from src.core.transformers.data import PipelineData
from src.domain.request.filters import QueryFilter

## Параметры

`site_config` — тот же файл, что использует CLI. Он содержит URL, pinned image,
profile/recipe, corpus path и collection; образ соответствует проверенной GPU.
Дополнительные source/retrieval overrides задаются явно в `site_overrides`.

`build_collection=True` разрешает создать новую generation в указанном site config
physical name. Занятая коллекция будет отклонена; выбирайте новое имя заранее.
При готовой коллекции оставьте `False`. Для представления документа и tokenizer
аудита нужен доступ к исходному корпусу. Новый output directory выбирается для
каждого опыта; существующие JSON/CSV не перезаписываются.


In [ ]:
site_config = Path("/srv/recsys/config/vector-site.yaml")
site_overrides = []
text = "Сварщик без опыта"
input_path = ROOT / "examples/blue-collar/queries.csv"
output_dir = ROOT / "outputs" / "vector-notebook" / uuid4().hex
build_collection = False

## Hydra-композиция

Context manager позволяет выполнять ячейку повторно. Каталог site config служит
основным, а search path добавляет базовые groups checkout. После изменения
параметров пересоберите объекты. Exporters здесь отключены, чтобы Compose API
не зависел от CLI runtime Hydra; пути экспорта будут заданы явно ниже.


In [ ]:
def compose_vector(wrapper):
    with initialize_config_dir(
        config_dir=str(site_config.resolve().parent), version_base="1.3"
    ):
        return compose(
            config_name=site_config.stem,
            overrides=[
                f"hydra.searchpath=[file://{ROOT / 'configs'}]",
                f"experiments/vacancy-search/vector@experiment={wrapper}",
                *site_overrides,
                "experiment.exporters=[]",
            ],
        )


cfg = compose_vector("002-interactive")
experiment = instantiate(cfg.experiment, _convert_="all")
leaf = experiment.registry["vector_search"]

## Подготовка запроса без HTTP

`prepare` нормализует query, добавляет `query: ` и строит declared scalar filters.
Здесь нет inference, fake vector или изменения PipelineData. План включает
context/filters текущего эксперимента. Дополнительные ручные условия следует
включать и в план, и в последующий `search`.


In [ ]:
probe = PipelineData(
    context=instantiate(cfg.experiment.context, _convert_="all"),
    filters=[*instantiate(cfg.experiment.filters, _convert_="all"), QueryFilter(text)],
)
plan = leaf.prepare(probe)
print(json.dumps(plan.to_dict(), ensure_ascii=False, indent=2))

## Представление документа и полный tokenizer audit

Reader проверяет snapshot корпуса, renderer выбирает поля по recipe, а embedder
добавляет `passage: `. Исходные fields сохраняются. Byte limit и token limit —
разные границы: query >512 tokens отвергается, document обрезается справа на TEI.

Аудит вызывает `/tokenize` с prefix и special tokens, по одному документу за раз,
без truncation. Отдельный response budget берётся из Hydra (16 MiB по умолчанию).
Список tokens освобождается до следующего запроса; сохраняются только counts.
Ошибка или oversized response прерывает ячейку без успешного audit report.
Эти counts описывают усечение, а не качество поиска.


In [ ]:
reader = instantiate(cfg.corpus.reader, _convert_="all")
renderer = instantiate(cfg.corpus.renderer, _convert_="all")
documents = reader.read()
example_document = documents[0]
example_text = renderer.render(example_document)
example_model_input = leaf.embedder.prepare_documents((example_text,))[0]
print(
    json.dumps(
        {
            "document_id": example_document.id,
            "model_input": example_model_input,
            "utf8_bytes": len(example_model_input.encode("utf-8")),
        },
        ensure_ascii=False,
        indent=2,
    )
)

In [ ]:
audit_report = None
leaf.embedder.check_runtime()
audit_counts = []
token_preview = []
for document in documents:
    document_text = renderer.render(document)
    model_input = leaf.embedder.prepare_documents((document_text,))[0]
    tokens = leaf.embedder.integration.tokenize((model_input,))[0]
    token_count = len(tokens)
    if not audit_counts:
        token_preview = tokens[:8]
    del tokens
    audit_counts.append({"document_id": document.id, "tokens": token_count})

truncated_count = sum(
    row["tokens"] > leaf.embedder.profile.max_tokens for row in audit_counts
)
audit_report = {
    "documents": len(audit_counts),
    "truncated_documents": truncated_count,
    "truncation_share": truncated_count / len(audit_counts),
    "max_tokens": max(row["tokens"] for row in audit_counts),
    "model_limit": leaf.embedder.profile.max_tokens,
}
print(
    json.dumps({"first_document_tokens": token_preview}, ensure_ascii=False, indent=2)
)
audit_report

## Необязательная подготовка новой коллекции

При `build_collection=True` эта ячейка выполняет HTTP writes в свежую generation:
полный preflight, indexes, encode/upload batches, read-back и публикацию ready.
После успеха receipt сохраняется отдельно. Ошибка записи receipt не отменяет
remote ready и не удаляет коллекцию. Повторный build того же name отклоняется.


In [ ]:
if build_collection:
    index_cfg = compose_vector("001-index")
    index_cfg.experiment.exporters = [
        {
            "_target_": "src.components.reporting.exporters.json.JSONExporter",
            "path": str(output_dir / "vector-index.json"),
        }
    ]
    output_dir.mkdir(parents=True, exist_ok=True)
    index = instantiate(index_cfg.experiment, _convert_="all")
    receipt = index.run()
    for exporter in index.exporters:
        exporter.export(receipt)
    print("Ready collection:", index.collection)

## Явный inference и native body

Сначала проверяется ready descriptor, затем явно выполняется один query encode.
`plan.body` принимает полученный vector и не делает inference. Полный `native_body`
доступен в kernel; вывод показывает первые coordinates и размер. Следующая ячейка
отправляет ровно этот body и возвращает candidate pool до TopK.


In [ ]:
leaf.integration.check_collection(leaf.collection, leaf.source)
query_embedding = leaf.embedder.encode_query(plan.original_query)
native_body = plan.body(query_embedding)
print(
    json.dumps(
        {
            "profile_id": query_embedding.profile_id,
            "dimensions": len(query_embedding.values),
            "vector_head": list(query_embedding.values[:8]),
            "request_settings": {
                key: value for key, value in native_body.items() if key != "query"
            },
        },
        ensure_ascii=False,
        indent=2,
    )
)

In [ ]:
candidates = leaf.integration.query(leaf.collection, native_body, leaf.source)
[
    {"id": item.id, "score": item.score, "fields": item.fields}
    for item in candidates.items
]

## Обычный end-to-end поиск

`SearchExperiment.search` повторяет полный путь с новым inference и применяет
настроенный TopK. Дополнительные context/filters передаются обычными аргументами
`search(text, context={...}, filters=[...])`; query остаётся QueryFilter.
Оценка semantic relevance, ANN recall и performance требует отдельных измерений.


In [ ]:
serp = experiment.search(text)
rows = [
    {"id": item.id, "score": item.score, "fields": item.fields} for item in serp.items
]
rows

## CSV batch и явное сохранение

Wrapper `003-batch` использует прежний `id,text` CSV и тот же собранный experiment.
Весь CSV проверяется до первого запроса; ошибка любого поиска не создаёт частичный
export. `run()` возвращает artifact, затем CSV сохраняется отдельной операцией.
Формат выдач прежний: query id/text, rank, document id, score и полный JSON fields.


In [ ]:
batch_cfg = compose_vector("003-batch")
batch_cfg.experiment.input_path = str(input_path)
batch_cfg.experiment.exporters = [
    {
        "_target_": "src.components.reporting.exporters.csv.SearchCSVExporter",
        "path": str(output_dir / "searches.csv"),
    }
]
batch = instantiate(batch_cfg.experiment, search=experiment, _convert_="all")
artifact = batch.run()
artifact.to_rows()[:5]

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
for exporter in batch.exporters:
    exporter.export(artifact)
output_dir / "searches.csv"